# 03 Validation

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/Gates Foundation/Building Dataset Validation/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Gates Foundation/Building Dataset Validation")
CONFIG_PATH  = PROJECT_ROOT / "configs/validation_configs.yaml"

# Set overwrite=True to re-run cities whose outputs already exist.
# When False (default), cities with existing sentinel parquets are skipped automatically.
OVERWRITE = True

In [3]:
import sys, os
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
import logging
import yaml
from src.validator import UrbanValidator

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

# Patch overwrite flag into config before constructing Validator
with open(CONFIG_PATH) as f:
    _cfg_preview = yaml.safe_load(f)

datasets_preview = _cfg_preview.get("vector", {}).get("datasets", [])
enabled = [d["name"] for d in datasets_preview if d.get("enabled", True)]
print(f"Config: {CONFIG_PATH}")
print(f"Enabled candidate datasets: {enabled}")

Config: /content/drive/MyDrive/Gates Foundation/Building Dataset Validation/configs/validation_configs.yaml
Enabled candidate datasets: ['overture', 'gba', 'globfp']


In [5]:
# Instantiate the validator — this reads the config and AOI tracker,
# resolves file paths, and logs how many datasets are queued.
# The overwrite flag from the cell above is injected before loading.
import yaml, copy

with open(CONFIG_PATH) as f:
    _cfg_patched = yaml.safe_load(f)
_cfg_patched.setdefault("output", {})["overwrite"] = OVERWRITE

# Write patched config to a temp file so Validator can read it
import tempfile, os
_tmp = tempfile.NamedTemporaryFile(
    mode="w", suffix=".yaml", delete=False, dir=PROJECT_ROOT / "configs"
)
yaml.dump(_cfg_patched, _tmp)
_tmp.close()
_PATCHED_CONFIG_PATH = _tmp.name

v = UrbanValidator(_PATCHED_CONFIG_PATH)
print(f"\nDatasets queued: {len(v.datasets)}")
for ds in v.datasets:
    print(f"  {ds['id']}")

20:05:58  INFO      Validation tracker: 1 -> 1 suitable rows.
20:05:58  INFO      Loaded 1 dataset(s) for validation.
20:05:58  INFO      Loaded 1 dataset(s) for validation.


[capetown_aoi.geojson] fixing 1 invalid geometries...

Datasets queued: 1
  zaf-capetown


In [6]:
import pandas as pd

results = v.validate_vector()

# Clean up temp config
try:
    os.unlink(_PATCHED_CONFIG_PATH)
except Exception:
    pass

# Summary
summary = pd.DataFrame(
    [{"aoi": k, "status": "ok" if v else "failed"} for k, v in results.items()]
)
print(f"\nDone — {len(summary)} AOIs processed.\n")
display(summary.groupby("status")["aoi"].count().rename("count").to_frame())
display(summary)

20:05:58  INFO      ━━━━  zaf-capetown  ━━━━
20:05:58  INFO      MEM [zaf-capetown start] RSS = 296 MB
20:05:59  INFO      [zaf-capetown] Working CRS: EPSG:32734 | 175 tiles.
20:05:59  INFO      Created 175 records
20:05:59  INFO      [zaf-capetown] AOI dissolved area: 14.800 km².


[captown_UoE.geojson] fixing 87 invalid geometries...


20:06:06  INFO      [zaf-capetown] Reference buildings: 80814 (from 1 file(s))
20:06:06  INFO      [zaf-capetown / overture] Candidate: zaf_capetown_overture_building.parquet
20:06:08  INFO      [zaf-capetown / overture] Candidate buildings: 86061
20:10:43  INFO      [zaf-capetown / overture] Tile metrics saved → vector_metrics_tiles_overture.parquet
20:10:43  INFO      [zaf-capetown / overture] Per-size-bin metrics saved → vector_size_bin_metrics_overture.parquet
20:10:43  INFO      MEM [zaf-capetown/overture done] RSS = 864 MB
20:10:43  INFO      [zaf-capetown / gba] Candidate: zaf_capetown_gba.parquet
20:10:45  INFO      [zaf-capetown / gba] Candidate buildings: 108432
20:17:00  INFO      [zaf-capetown / gba] Tile metrics saved → vector_metrics_tiles_gba.parquet
20:17:00  INFO      [zaf-capetown / gba] Per-size-bin metrics saved → vector_size_bin_metrics_gba.parquet
20:17:01  INFO      MEM [zaf-capetown/gba done] RSS = 1243 MB
20:17:01  INFO      [zaf-capetown / globfp] Candidate: z


Done — 1 AOIs processed.



,count
status,
ok,1


,aoi,status
0,zaf-capetown,ok


In [7]:
summary.groupby("status")["aoi"].count().rename("count").to_frame()

,count
status,
ok,1


In [8]:
summary

,aoi,status
0,zaf-capetown,ok
